# FDIC Group 14 — CSV-based four-model pipeline

**Sources:** supplied MLR, Decision Tree, Random Forest, and XGBoost notebooks; Group 14 presentation.

**Expected input:** a single Q1 bank-year CSV containing `CERT`, `year`, `TARGET_EEFFR_NEXT_Q1`, financial predictors, `SPECGRPDESC`, and preferably current-year `EEFFR`. One row per bank/year. The target is **next-year Q1 EEFFR**, not the same-year EEFFR. All ratios are in EEFFR percentage points. Do not derive target from current EEFFR in the same row.

**Default chronological split:** train 2022–2024, holdout test 2025, out-of-time score 2026. Change the year settings below if needed. All learned imputations, feature selection and winsorization are fit using training data only.

**Caveat:** these are reusable refactorings of the notebooks and final deck specifications, not bit-for-bit reproductions of every original tuning/search experiment. The number of selected features may differ with a new CSV. No claimed deck metrics are prefilled.

In [ ]:
# Run from the same directory as the five .py files.
# Install once in your environment if necessary:
# %pip install pandas numpy scikit-learn xgboost matplotlib
from pathlib import Path
import pandas as pd
import numpy as np
from feature_engineering import read_csv, split_by_year, evaluate, TARGET, expanding_folds
from model_mlr import train_model as fit_mlr
from model_decision_tree import train_model as fit_tree
from model_random_forest import train_model as fit_rf
from model_xgboost import train_model as fit_xgb

CSV_PATH = Path('fdic_all_q1.csv')  # <-- change to your combined CSV
OUTPUT_DIR = Path('fdic_results')
TRAIN_YEARS = (2022, 2023, 2024)
TEST_YEAR = 2025
OOT_YEAR = 2026
RUN_EXPANDING_CV = False  # True: two chronological validation folds; can take time
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

## 1. Load CSV and create train/test/out-of-time splits

In [ ]:
df = read_csv(CSV_PATH)
train, test, oot = split_by_year(df, TRAIN_YEARS, TEST_YEAR, OOT_YEAR)
print('All:',df.shape,'Train:',train.shape,'Test:',test.shape,'OOT:',oot.shape)
print('Train years:',sorted(train.year.unique()),'Test years:',sorted(test.year.unique()))
print('Available current-year EEFFR:', 'EEFFR' in df.columns)
display(df.head(3))

## 2. Fit all four models and evaluate untouched holdout

In [ ]:
fitters = {'MLR':fit_mlr, 'Decision Tree':fit_tree, 'Random Forest':fit_rf, 'XGBoost':fit_xgb}
models, metrics, prediction_table = {}, [], test[['CERT','year',TARGET]].copy()
for name, fit in fitters.items():
    print('Training:',name)
    model = fit(train)
    pred = model.predict(test)
    models[name] = model
    prediction_table[name+'_pred'] = pred
    result = {'Model':name, 'features':len(model.feature_names), **evaluate(test[TARGET],pred)}
    metrics.append(result)
    print(result)
results = pd.DataFrame(metrics).sort_values('MAE')
display(results)
results.to_csv(OUTPUT_DIR/'holdout_metrics.csv',index=False)
prediction_table.to_csv(OUTPUT_DIR/'test_predictions.csv',index=False)

## 3. Naive benchmark and retrospective subgroups

In [ ]:
if 'EEFFR' in test:
    valid = test['EEFFR'].notna()
    if valid.any():
        print('Naive benchmark (next-year EEFFR = current EEFFR):',evaluate(test.loc[valid,TARGET],test.loc[valid,'EEFFR']))
for cap in (100,200):
    mask = (test[TARGET]>0)&(test[TARGET]<=cap)
    print(f'Actual next-year EEFFR <= {cap} (retrospective only):',int(mask.sum()))
    for name in models:
        print(name,evaluate(test.loc[mask,TARGET],prediction_table.loc[mask,name+'_pred']))

## 4. Optional expanding-window validation (fit selection independently in each fold)

In [ ]:
cv_rows=[]
if RUN_EXPANDING_CV:
    for tr,va,year in expanding_folds(train):
        for name,fit in fitters.items():
            print('CV:',name,'validation year:',year)
            try:
                fitted=fit(tr)
                cv_rows.append({'Model':name,'validation_year':year,'features':len(fitted.feature_names),
                                **evaluate(va[TARGET],fitted.predict(va))})
            except Exception as exc:
                print('CV failed:',name,year,repr(exc))
    if cv_rows:
        cv_table=pd.DataFrame(cv_rows)
        display(cv_table)
        cv_table.to_csv(OUTPUT_DIR/'expanding_window_cv.csv',index=False)

## 5. Score out-of-time Q1 2026 rows (next-year Q1 2027)

In [ ]:
if len(oot):
    forecast=oot[['CERT','year']].copy()
    if 'NAME' in oot: forecast['NAME']=oot['NAME']
    if 'EEFFR' in oot: forecast['current_EEFFR']=oot['EEFFR']
    for name,model in models.items():
        forecast[name+'_pred_next_Q1_EEFFR']=model.predict(oot)
        if 'current_EEFFR' in forecast:
            forecast[name+'_flag_rise_gt_10pp']=(forecast[name+'_pred_next_Q1_EEFFR']-forecast['current_EEFFR'])>10
    display(forecast.head())
    forecast.to_csv(OUTPUT_DIR/'oot_2027_forecasts.csv',index=False)
else:
    print('No out-of-time rows for',OOT_YEAR)

## Notes on reproducibility

- **MLR:** log(next-year EEFFR), financial ratios, 1–99% train-only winsorization, specialization dummy encoding and standardized OLS; the original notebook also evaluated raw OLS, Ridge, Lasso and Huber. The deck's 24-feature count depends on available categories and original input.
- **Decision Tree:** original core/secondary variable lists, 70% observed threshold, near-duplicate threshold 0.9999, permutation top-10 ranking, max depth 10, leaf 100, log1p target.
- **Random Forest:** mutual information selection of 25 inputs, 200 trees, leaf 5, max_features 0.5, log1p target; shared engineering includes the ratio definitions. The refactor encodes specialization as a single ordinal input rather than the original one-hot representation, so results can differ.
- **XGBoost:** original 14 ratio definitions, log assets, correlation threshold 0.95, pseudo-Huber objective and 0.001 importance threshold. This implementation uses train-only median imputation, while the source XGBoost notebook used XGBoost's native handling of missing values; resulting metrics may differ.
- **Do not use current EEFFR as a predictor in tree/RF/XGBoost:** it is excluded in this implementation, consistent with the presentation's shared modeling description. MLR's original notebook explicitly included current EEFFR in its final forecasting comparison, which differs from the deck's shared description.
- Original SQL/S3 acquisition and model artifact persistence are intentionally not part of these six files. This notebook reads a local CSV and writes local CSV results.